In [5]:
import numpy as np
import pygame
from scipy.integrate import solve_bvp
import multiprocessing
multiprocessing.set_start_method('spawn' , force = True)
import os
from stable_baselines3 import PPO ,A2C , DDPG ,TD3 , SAC
from stable_baselines3.common.vec_env import DummyVecEnv , SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.evaluation import evaluate_policy
import time
import gymnasium as gym
from gymnasium.spaces import Discrete, Box, Dict
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
import math
from stable_baselines3.common.callbacks import EvalCallback
import optuna

In [6]:
import sys
print(sys.version)

3.12.1 (main, Sep 30 2024, 17:05:21) [GCC 9.4.0]


In [12]:
class Elastica_env(gym.Env):
    def __init__(self):
        super(Elastica_env, self).__init__()
        self.action_space = Box(low=np.array([-0.2, -0.2], dtype=np.float32), high=np.array([0.1 , 0.2], dtype=np.float64))
        self.observation_space = Box(low=np.float32(-100), high=np.float32(100), shape=(13,), dtype=np.float64)
        self.target = Box(low=np.array([4.5, -1], dtype=np.float32), high=np.array([5.57, 1], dtype=np.float64))
        self.num_timestep = 0
        self.reward = 0
        self.x = []
        self.y = []
        self.screen_width = 800.0
        self.screen_height = 600.0
        self.zoom_factor = 60.0
        self.enable_render = False
        self.h = -0.4
        self.v = 0.15

    # Add the seed() method
    def seed(self, seed=None):
        self.np_random, seed = gym.utils.seeding.np_random(seed)
        return seed
    def step(self, action):
        self.num_timestep += 1
        self.h += action[0]
        self.v += action[1]
        self.X, self.Y, self.theta_dash_0  ,self.theta_dash_l, self.theta_l , self.E = self.elastica(self.h, self.v)

        new_observation = self.get_observation()
        self.render(self.enable_render)
        self.reward = self.score()
        done = self.get_done()
        truncation = self.get_truncation()
        info = {}
        return new_observation, self.reward, done, truncation, info

    def elastica(self, h, v):
        l = 6
        s = np.linspace(0, l, 500)
        def f(s, y):
            return np.vstack((y[1], h * np.sin(y[0]) - v * np.cos(y[0])))
        def bc(ya, yb):
            return np.array([ya[0] - 0, yb[1] - 0])
        y0 = np.zeros((2, s.size))
        sol = solve_bvp(f, bc, s, y0)
        theta = sol.sol(s)[0]
        dtheta_ds = sol.sol(s)[1]
        y1 = np.cos(theta)
        y2 = np.sin(theta)
        y3 = (dtheta_ds)**2
        self.x = []
        self.y = []
        for i in range(len(s)):
            self.x.append(np.trapezoid(y1[:i+1], x=s[:i+1]))
            self.y.append(np.trapezoid(y2[:i+1], x=s[:i+1]))

        e = 0.5*(np.trapezoid(y3 , s))-v*(np.trapezoid(y2 , s)) + h*(np.trapezoid(1-y1 , s))

        return self.x, self.y, dtheta_ds[0] ,dtheta_ds[-1] , theta[-1] , e 

    def get_observation(self):
        self.x_tip = self.X[-1]
        self.y_tip = self.Y[-1]
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        return np.array([self.x_tip, self.y_tip, self.X[200], self.Y[200], self.X[400], self.Y[400], 
                         self.theta_l , self.theta_dash_0 , self.theta_dash_l ,self.E  ,
                         self.x_target, self.y_target ,d] ,  dtype=np.float64)

    def score(self):
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        #d_score = -(d)**2
        d_score = math.exp(-d)
#         if d>0 and d<0.002:
#             d_score += 1
#         elif d>0 and d<0.05:
#             d_score += 0.3
#         elif d>0 and d>0.1:
#             d_score += 0.1
        

        total_score = d_score
        return total_score

    def get_done(self):
        done = False
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        if d < 0.002:
            done = True
        return done

    def get_truncation(self):
        truncation = False
        if self.num_timestep > 19 :
            truncation = True
        return truncation

    def reset(self, seed=None):
        if seed is not None:
            self.np_random, seed = gym.utils.seeding.np_random(seed)
        targ = self.target.sample()
        self.x_target = targ[0]
        self.y_target = targ[1]
        if self.y_target<=0:
            self.h = -0.4
            self.v = 0.15
        if self.y_target>0:
            self.h = -0.4
            self.v = -0.15

        self.X, self.Y, self.theta_dash_0 ,self.theta_dash_l , self.theta_l , self.E  = self.elastica(self.h, self.v)
        self.num_timestep = 0
        self.reward = 0
        observation = self.get_observation()
        info = {}
        return observation, info

    def render(self, enable_render):
        if not enable_render:
            return
        pygame.init()
        screen = pygame.display.set_mode((int(self.screen_width), int(self.screen_height)))
        pygame.display.set_caption("Elastica")
        screen.fill((255, 255, 255))
        offset_x = (self.screen_width - 10 * self.zoom_factor) / 2
        offset_y = (self.screen_height - 1.5 * self.zoom_factor) / 2
        points = [(self.X[i], self.Y[i]) for i in range(len(self.X))]
        pygame.draw.lines(screen, (0, 0, 0), False, [(point[0] * self.zoom_factor + offset_x, point[1] * self.zoom_factor + offset_y) for point in points])
        pygame.draw.line(screen, (255, 0, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x + 50 * np.sign(self.h), (self.Y[-1]) * self.zoom_factor + offset_y), 3)
        pygame.draw.line(screen, (0, 255, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y + 50 * np.sign(self.v)), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y + 25), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y - 25), 3)
        pygame.draw.circle(screen, (255, 0, 0), (int(self.x_target * self.zoom_factor + offset_x), int(self.y_target * self.zoom_factor + offset_y)), 5)
        font = pygame.font.Font(None, 36)
        score_text = font.render(f"Timesteps: {self.num_timestep}", True, (0, 0, 0))
        screen.blit(score_text, (int(self.screen_width - score_text.get_width() - 30), 120))
        pygame.display.flip()

    def close(self):
        pygame.quit()


In [13]:
import pickle

env = Elastica_env()
try:
    pickle.dumps(env)
    print("Environment is serializable")
except Exception as e:
    print("Environment is not serializable:", e)


Environment is serializable


In [14]:
num_cpu_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores available: {num_cpu_cores}")

Number of CPU cores available: 2


In [15]:
env = Elastica_env()

In [16]:
check_env(env)
env.close()

In [17]:
env.enable_render = True
episodes = 10
for episode in range(1, episodes+1):
    state , info = env.reset()
    done = False
    score = 0
    truncation=False
    while not (done or truncation):
        #env.render(True )
        action = env.action_space.sample()
        #print(action)
        n_state, reward, done , truncation ,info = env.step(action)
        score+=reward
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

Episode:1 Score:2.3078816463769094
Episode:2 Score:4.463309827416999
Episode:3 Score:1.140784286023238
Episode:4 Score:5.752759552142582
Episode:5 Score:8.490559956383004
Episode:6 Score:8.407058559318047
Episode:7 Score:3.2328005210104496
Episode:8 Score:6.087516976081203
Episode:9 Score:3.8245043769987532
Episode:10 Score:6.117561076477546


In [18]:
# Function to create and seed each environment
def make_custom_env(rank: int, seed: int = 0):
    def _init():
        env = Elastica_env()  
        env.seed(seed + rank)
        return env
    return _init

# Set the multiprocessing start method (recommended on Windows)
multiprocessing.set_start_method('spawn', force=True)

# Set up the number of environments
num_cpu = 2  # Number of environments to run in parallel (number of CPU cores)
env2 = SubprocVecEnv([make_custom_env(i) for i in range(num_cpu)], start_method='spawn')
eval_env1 = SubprocVecEnv([make_custom_env(i) for i in range(num_cpu)], start_method='spawn')


# Initialize and train the RL agent 
# model = SAC("MlpPolicy", env2, verbose=1)
# model.learn(total_timesteps=200)


In [19]:
env.enable_render = False
log_path = os.path.join('Training' , 'Logs')
os.makedirs(os.path.dirname(log_path) , exist_ok = True)
eval_env = Elastica_env()
eval_env = Monitor(eval_env , log_path)
#policy_kwargs = dict(net_arch=dict(pi=[256, 256 , 256], qf=[500, 400 , 400]) )  # by default tanh activation function
#policy_kwargs = dict(activation_fn=th.nn.ReLU,net_arch=dict(pi=[32, 32], qf=[32, 32]))    # ReLu activation function

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate' , 1e-5 , 1e-3 ,log = True)
    entropy_coef = trial.suggest_float ('entropy_coef' , 1e-4 , 1e-2  , log = True)
    gamma = trial.suggest_float('gamma' ,0.9 ,0.9999 , log = True)
    tau = trial.suggest_float('tau' , 0.005 , 0.05 , log = True)
    batch_size = trial.suggest_categorical('batch_size' , [64 , 128 , 256 , 1000 , 2000 , 4000 , 8000 , 16000])
    target_update_interval = trial.suggest_categorical('target_update_interval' , [1000 , 5000 , 10000])
    gradient_steps = trial.suggest_categorical('gradient_steps' , [1 , 2, 4])
    buffer_size = trial.suggest_categorical('buffer_size' , [100000 , 200000 , 500000 , 1000000 , 2000000])
    
    model = SAC('MlpPolicy', env2, verbose=0 , learning_rate = learning_rate ,
                ent_coef = entropy_coef , gamma = gamma , tau = tau , batch_size = batch_size , 
                target_update_interval = target_update_interval , gradient_steps= gradient_steps ,
                buffer_size = buffer_size )
    
    eval_callback = EvalCallback(eval_env ,  eval_freq = 100 ,deterministic = True )
    model.learn(total_timesteps = 200 , callback = eval_callback)

    mean_reward , _ = evaluate_policy(model , eval_env , n_eval_episodes = 10 ,deterministic=True)
    return mean_reward

study = optuna.create_study (direction = 'maximize')
study.optimize(objective , n_trials = 2 , n_jobs = 1)
best_parameters = study.best_params
print(best_parameters)

[I 2024-10-19 06:34:09,218] A new study created in memory with name: no-name-e002a3be-57c9-4b9f-bb7d-d76bd307606c
/home/codespace/.python/current/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x787ccf5be6c0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x787c895fa330>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Eval num_timesteps=200, episode_reward=1.83 +/- 0.44
Episode length: 20.00 +/- 0.00
New best mean reward!


[I 2024-10-19 06:34:20,433] Trial 0 finished with value: 1.9390867999999997 and parameters: {'learning_rate': 1.2059389425962598e-05, 'entropy_coef': 0.0011583760844285423, 'gamma': 0.9517764586888205, 'tau': 0.006827371189730593, 'batch_size': 256, 'target_update_interval': 1000, 'gradient_steps': 1, 'buffer_size': 500000}. Best is trial 0 with value: 1.9390867999999997.
/home/codespace/.python/current/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x787ccf5be6c0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x787c8920e3c0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Eval num_timesteps=200, episode_reward=1.70 +/- 0.52
Episode length: 20.00 +/- 0.00
New best mean reward!


[I 2024-10-19 06:35:01,554] Trial 1 finished with value: 1.5090942999999999 and parameters: {'learning_rate': 2.365858913024466e-05, 'entropy_coef': 0.0001270348638078619, 'gamma': 0.9864498925483983, 'tau': 0.020221377836865347, 'batch_size': 8000, 'target_update_interval': 10000, 'gradient_steps': 2, 'buffer_size': 500000}. Best is trial 0 with value: 1.9390867999999997.


{'learning_rate': 1.2059389425962598e-05, 'entropy_coef': 0.0011583760844285423, 'gamma': 0.9517764586888205, 'tau': 0.006827371189730593, 'batch_size': 256, 'target_update_interval': 1000, 'gradient_steps': 1, 'buffer_size': 500000}


In [20]:
log_path = os.path.join('Training', 'Logs')
#PPO_path = os.path.join('Training', 'Saved Models', 'DDPG')
#print(PPO_path)
#env = DummyVecEnv([lambda: env])
model = SAC( 'MlpPolicy', env2, verbose=0 ,  learning_rate=best_parameters['learning_rate'],
            ent_coef=best_parameters['entropy_coef'], gamma=best_parameters['gamma'], tau=best_parameters['tau'], 
           batch_size=best_parameters['batch_size'], target_update_interval = best_parameters['target_update_interval'],
           gradient_steps=best_parameters['gradient_steps'],buffer_size=best_parameters['buffer_size'])  

In [21]:
eval_callback = EvalCallback(eval_env,  eval_freq = 100 ,deterministic = True )
model.learn(total_timesteps=2000 , callback = eval_callback )

/home/codespace/.python/current/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x787ccf5be6c0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x787c8923e3c0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Eval num_timesteps=200, episode_reward=2.56 +/- 0.46
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=400, episode_reward=2.13 +/- 0.90
Episode length: 20.00 +/- 0.00
Eval num_timesteps=600, episode_reward=5.32 +/- 2.85
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=800, episode_reward=5.50 +/- 1.69
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=1000, episode_reward=3.16 +/- 1.52
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1200, episode_reward=4.75 +/- 1.72
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1400, episode_reward=7.31 +/- 3.33
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=1600, episode_reward=6.96 +/- 1.59
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1800, episode_reward=6.84 +/- 1.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=2000, episode_reward=6.99 +/- 2.06
Episode length: 20.00 +/- 0.00


In [9]:
SAC_path = os.path.join('Training', 'Saved Models', 'Elasticadesk10_SAC')   # Keep all the files in same directory
os.makedirs(os.path.dirname(SAC_path), exist_ok=True)

In [20]:
model.save(SAC_path)

In [11]:
model=SAC.load(SAC_path)
print(f"Model loaded from: {SAC_path}")

Model loaded from: Training\Saved Models\Elasticadesk10_SAC


C:\Users\Jayhind\anaconda3\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(


In [22]:
env.enable_render = True
episodes = 20
for episode in range(1, episodes+1):
    state  , info = env.reset()
    done = False
    score = 0 
    truncation=False
    test=[]
    n_score=[]
    dis = []
    while not (done or truncation):
        action,_ = model.predict(state , deterministic=True)
        #print(action)
        state, reward, done, truncation ,info = env.step(action)
        n_score.append(reward)
        dis.append(state[-1])
        score+=reward
        test.append(done)
    #print(test)
    print(len(test))
    print(n_score)
    #print(np.min(dis))
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

20
[0.33745763188522143, 0.15749041076966966, 0.26596624241899214, 0.24343113808660966, 0.245790172952058, 0.2572131778170142, 0.2357798501922551, 0.24509623736824332, 0.2441542948559075, 0.24365352172002164, 0.2436806667795345, 0.24428311919585075, 0.24602635950600868, 0.25257611638530236, 0.3132364252584333, 0.23679875503422915, 0.23791422116631364, 0.23943213284750223, 0.2404108174113942, 0.24115062282229904]
Episode:1 Score:4.97154191447286
20
[0.30101657728508585, 0.12835102269635193, 0.23428135353485252, 0.2088664344060633, 0.21043971017265478, 0.22024209092527816, 0.43493438567207815, 0.17822917740343766, 0.19687921919530815, 0.20292198150998444, 0.2056400103042989, 0.2074673227407889, 0.20968068306450066, 0.21562617353625155, 0.2658145203888009, 0.19916669726216626, 0.20226100751544457, 0.20406476991648398, 0.2051513381131854, 0.20593082108001076]
Episode:2 Score:4.436965296723026
20
[0.3769743815526251, 0.44136451535080307, 0.41882834716078227, 0.4638037297844477, 0.4623445878

In [22]:
env.close()

In [21]:
import numpy as np
from scipy.integrate import solve_bvp
import gymnasium as gym
from gymnasium.spaces import Box
import numba
import pygame
import pickle
import math
from scipy.spatial import cKDTree

class Elastica_env(gym.Env):
    def __init__(self):
        super(Elastica_env, self).__init__()
        self.action_space = Box(low=np.array([-0.2, -0.2], dtype=np.float32), high=np.array([0.1 , 0.2], dtype=np.float64))
        self.observation_space = Box(low=np.float32(-100), high=np.float32(100), shape=(13,), dtype=np.float64)
        
        # Update target space to match cheat sheet
        self.target = Box(low=np.array([0.5, -0.4], dtype=np.float32), high=np.array([0.9, 0.4], dtype=np.float64))
        self.num_timestep = 0
        self.reward = 0
        self.x = []
        self.y = []
        self.screen_width = 800.0
        self.screen_height = 600.0
        self.zoom_factor = 60.0
        self.enable_render = False
        self.h = -0.4
        self.v = 0.15

        # Pre-allocate arrays for elastica calculation
        self.s = np.linspace(0, 6, 500)
        self.y0 = np.zeros((2, self.s.size))

        # Load the cheat sheet data
        with open('cheat_sheet_data.pkl', 'rb') as f:
            self.cheat_sheet_data = pickle.load(f)
        
        # Create KD-tree for efficient nearest neighbor search
        self.kdtree = cKDTree(self.cheat_sheet_data['grid_points'])

    # Add the seed() method
    def seed(self, seed=None):
        self.np_random, seed = gym.utils.seeding.np_random(seed)
        return seed
    def step(self, action):
        self.num_timestep += 1
        self.h += action[0]
        self.v += action[1]
        self.X, self.Y, self.theta_dash_0  ,self.theta_dash_l, self.theta_l , self.E = self.elastica(self.h, self.v)

        new_observation = self.get_observation()
        self.render(self.enable_render)
        self.reward = self.score()
        done = self.get_done()
        truncation = self.get_truncation()
        info = {}
        return new_observation, self.reward, done, truncation, info

    def elastica(self, h, v):
        l = 1
        s = self.s

        # Find the nearest neighbor in the cheat sheet
        _, nearest_index = self.kdtree.query([h, v])
        
        # Get the initial guess from the cheat sheet
        theta_guess = self.cheat_sheet_data['theta_values'][nearest_index]
        theta_prime_guess = self.cheat_sheet_data['theta_prime_values'][nearest_index]
        
        # Create the initial guess array
        y0 = np.array([theta_guess, theta_prime_guess])

        def f(s, y):
            return np.vstack((y[1], h * np.sin(y[0]) - v * np.cos(y[0])))

        def bc(ya, yb):
            return np.array([ya[0], yb[1]])

        sol = solve_bvp(f, bc, s, y0)
        theta = sol.sol(s)[0]
        dtheta_ds = sol.sol(s)[1]
        y1 = np.cos(theta)
        y2 = np.sin(theta)
        y3 = (dtheta_ds)**2

        x = np.cumsum(y1) * (l / 500)
        y = np.cumsum(y2) * (l / 500)

        e = 0.5 * np.sum(y3) * (l / 500) - v * np.sum(y2) * (l / 500) + h * np.sum(1 - y1) * (l / 500)

        return x, y, dtheta_ds[0], dtheta_ds[-1], theta[-1], e

    def get_observation(self):
        self.x_tip = self.X[-1]
        self.y_tip = self.Y[-1]
        d = math.hypot(self.x_tip - self.x_target, self.y_tip - self.y_target)
        return np.array([self.x_tip, self.y_tip, self.X[200], self.Y[200], self.X[400], self.Y[400], 
                         self.theta_l, self.theta_dash_0, self.theta_dash_l, self.E,
                         self.x_target, self.y_target, d], dtype=np.float64)

    def score(self):
        d = math.hypot(self.x_tip - self.x_target, self.y_tip - self.y_target)
        return math.exp(-d)

    def get_done(self):
        done = False
        d = math.hypot(self.x_tip - self.x_target, self.y_tip - self.y_target)
        if d < 0.002:
            done = True
        return done

    def get_truncation(self):
        truncation = False
        if self.num_timestep > 19 :
            truncation = True
        return truncation

    def reset(self, seed=None):
        if seed is not None:
            self.np_random, seed = gym.utils.seeding.np_random(seed)
        targ = self.target.sample()
        self.x_target = targ[0]
        self.y_target = targ[1]
        if self.y_target<=0:
            self.h = -0.4
            self.v = 0.15
        if self.y_target>0:
            self.h = -0.4
            self.v = -0.15

        self.X, self.Y, self.theta_dash_0 ,self.theta_dash_l , self.theta_l , self.E  = self.elastica(self.h, self.v)
        self.num_timestep = 0
        self.reward = 0
        observation = self.get_observation()
        info = {}
        return observation, info

    def render(self, enable_render):
        if not enable_render:
            return
        pygame.init()
        screen = pygame.display.set_mode((int(self.screen_width), int(self.screen_height)))
        pygame.display.set_caption("Elastica")
        screen.fill((255, 255, 255))
        offset_x = (self.screen_width - 10 * self.zoom_factor) / 2
        offset_y = (self.screen_height - 1.5 * self.zoom_factor) / 2
        points = [(self.X[i], self.Y[i]) for i in range(len(self.X))]
        pygame.draw.lines(screen, (0, 0, 0), False, [(point[0] * self.zoom_factor + offset_x, point[1] * self.zoom_factor + offset_y) for point in points])
        pygame.draw.line(screen, (255, 0, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x + 50 * np.sign(self.h), (self.Y[-1]) * self.zoom_factor + offset_y), 3)
        pygame.draw.line(screen, (0, 255, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y + 50 * np.sign(self.v)), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y + 25), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y - 25), 3)
        
        # Draw target box
        pygame.draw.rect(screen, (0, 0, 255), (
            0.5 * self.zoom_factor + offset_x,
            -0.4 * self.zoom_factor + offset_y,
            0.4 * self.zoom_factor,
            0.8 * self.zoom_factor
        ), 2)
        
        pygame.draw.circle(screen, (255, 0, 0), (int(self.x_target * self.zoom_factor + offset_x), int(self.y_target * self.zoom_factor + offset_y)), 5)
        font = pygame.font.Font(None, 36)
        score_text = font.render(f"Timesteps: {self.num_timestep}", True, (0, 0, 0))
        screen.blit(score_text, (int(self.screen_width - score_text.get_width() - 30), 120))
        pygame.display.flip()

    def close(self):
        pygame.quit()


In [22]:
import pickle

env = Elastica_env()
try:
    pickle.dumps(env)
    print("Environment is serializable")
except Exception as e:
    print("Environment is not serializable:", e)


Environment is serializable


In [23]:
num_cpu_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores available: {num_cpu_cores}")

Number of CPU cores available: 12


In [24]:
env = Elastica_env()

In [25]:
check_env(env)
env.close()

In [26]:
env.enable_render = True
episodes = 50
for episode in range(1, episodes+1):
    state  , info = env.reset()
    done = False
    score = 0 
    truncation=False
    test=[]
    n_score=[]
    dis = []
    while not (done or truncation):
        action,_ = model.predict(state , deterministic=True)
        #print(action)
        state, reward, done, truncation ,info = env.step(action)
        n_score.append(reward)
        dis.append(state[-1])
        score+=reward
        test.append(done)
    #print(test)
    print(len(test))
    print(n_score)
    #print(np.min(dis))
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

20
[0.9154503692075665, 0.8307523708548948, 0.8419304023129306, 0.850326996041277, 0.8547068926425125, 0.856171342551604, 0.8567172563128516, 0.8569562293721364, 0.8570790496259901, 0.857151460397442, 0.857194299741222, 0.8572189187964412, 0.8572312106522368, 0.8572341413165666, 0.8572296706230438, 0.8572193428201328, 0.8572041114474761, 0.8571846058438358, 0.8571612717089896, 0.8571345234986137]
Episode:1 Score:17.14925446576777
20
[0.6969830796445042, 0.6220505987824847, 0.5635854550786769, 0.5987801327920945, 0.6380736805043276, 0.690524482482906, 0.7064845058159351, 0.6833257529540876, 0.7108911171069684, 0.6732253259584804, 0.712912121361415, 0.6671054562660569, 0.7142058694242441, 0.6625057498952083, 0.7152628831593886, 0.6593753163226339, 0.7157708620522777, 0.6575246201398289, 0.7159839351205645, 0.6563817103585362]
Episode:2 Score:13.46095265522062
20
[0.9395064733897323, 0.860182364710188, 0.8651625478027012, 0.866305728167883, 0.8640647048171314, 0.8604063636177631, 0.857897

In [13]:
SAC_path = "Elasticadesk10_SAC"
model=SAC.load(SAC_path)
print(f"Model loaded from: {SAC_path}")

Model loaded from: Elasticadesk10_SAC


c:\Documents\Jayhind\Reinforcement-Learning-to-control-flexible-structures\venv\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(


In [15]:
env.enable_render = True
episodes = 50
for episode in range(1, episodes+1):
    state  , info = env.reset()
    done = False
    score = 0 
    truncation=False
    test=[]
    n_score=[]
    dis = []
    while not (done or truncation):
        action,_ = model.predict(state , deterministic=True)
        #print(action)
        state, reward, done, truncation ,info = env.step(action)
        n_score.append(reward)
        dis.append(state[-1])
        score+=reward
        test.append(done)
    #print(test)
    print(len(test))
    print(n_score)
    #print(np.min(dis))
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

Episode: 1, Steps: 20, Score: 19.1759564248278
Rewards: [0.9372664974006173, 0.9259036111717769, 0.9419140795580013, 0.92974158545761, 0.9534879875902512, 0.9579588148134928, 0.9624229406924595, 0.964691478538921, 0.9658677591732024, 0.9664757714001885, 0.9667868068903919, 0.9669438356338785, 0.967020539584855, 0.9670556696620496, 0.9670701093240173, 0.9670742241196979, 0.9670735711457622, 0.9670705675894801, 0.9670670600360293, 0.9670635150451139]
Episode: 2, Steps: 20, Score: 17.784360108862828
Rewards: [0.9124054777286931, 0.9266150456781177, 0.8942276259437749, 0.9049330375323351, 0.8775004484441428, 0.89647131205006, 0.8732035918444429, 0.8950763236528605, 0.872579956812505, 0.8948434224758632, 0.8725055208229228, 0.8948095364651152, 0.8724944988586769, 0.8948045320520432, 0.872492603264457, 0.894803926374629, 0.8724926814901017, 0.8948039429409921, 0.8724926814901017, 0.8948039429409921]
Episode: 3, Steps: 20, Score: 19.430475444127797
Rewards: [0.9949901664161322, 0.960048178664